In [1]:
import os
import pandas as pd
import tensorflow as tf
import numpy as np
from datetime import datetime
from working_data import clean_five_minute_data, add_partial_hour_ohlc, add_timing, add_partial_rsi, add_rsi, normalize_by_window, clean_hour_data, split_multiresolution_chunks, regression_label_df, normalize_partial_hour
from constants.global_constants import *
from modeler import create_regression_model
from regression_losses import compile_model_recommended, compile_model_lightweight

2025-07-24 20:13:30.366747: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-24 20:13:30.398759: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-07-24 20:13:31.015716: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
try:
    policy = tf.keras.mixed_precision.Policy('mixed_float16')
    tf.keras.mixed_precision.set_global_policy(policy)
    print(f"Mixed precision policy set: {policy.name}")
    
    # Verify it's working
    print(f"Compute dtype: {policy.compute_dtype}")  # Should be float16
    print(f"Variable dtype: {policy.variable_dtype}")  # Should be float32
except Exception as e:
    print(f"Could not enable mixed precision: {e}")

Mixed precision policy set: mixed_float16
Compute dtype: float16
Variable dtype: float32


In [3]:
starting_dir = "data/final_data"
working_path = "data/experimenting"
instrument = "GBPUSD#"

In [4]:
# df = pd.read_csv(f"{starting_dir}/{instrument}/five_minutes.csv")
# print(f"Processing {instrument}...")
# print(f"  5 minute Original data: {len(df)} rows")
# print(f"  5 minute Time range: {datetime.fromtimestamp(df['time'].min())} to {datetime.fromtimestamp(df['time'].max())}")
# df = clean_five_minute_data(df)
# print(f"  Cleaned data: {len(df)} rows")
# df = add_timing(df)
# df = add_partial_hour_ohlc(df)
# df = add_rsi(df)
# df = normalize_by_window(
#     df, 
#     window_size=NORMALIZING_WINDOW_SIZE, 
#     low_col='low',
#     high_col='high',
#     normalizing_cols=[
#         'open',
#         'high',
#         'low',
#         'close'
#     ],
#     label_cols=['open', 'close'])

# hour_df = pd.read_csv(f"{starting_dir}/{instrument}/hours.csv")
# print(f"  Hour Original data: {len(hour_df)} rows")
# print(f"  Hour Time range: {datetime.fromtimestamp(hour_df['time'].min())} to {datetime.fromtimestamp(hour_df['time'].max())}")
# hour_df = clean_hour_data(hour_df)
# print(f"  Cleaned data: {len(hour_df)} rows")
# hour_df = add_timing(hour_df)
# hour_df = add_rsi(hour_df)
# hour_df = normalize_by_window(
#     hour_df, 
#     window_size=NORMALIZING_WINDOW_SIZE, 
#     low_col='low',
#     high_col='high',
#     normalizing_cols=[
#         'open',
#         'high',
#         'low',
#         'close'
#     ],
#     label_cols=['open', 'close'],
#     add_partial_hour=True)

# print('Adding Partial RSI...')

# df = add_partial_rsi(df, hour_df)

# print('Done')

# df = normalize_partial_hour(df, hour_df)

# print(f"Labeling...\n\n\n")
# df = regression_label_df(df, window_size=REGRESSION_LABELING_WINDOW_SIZE, 
#                 positive_slope=POSITIVE_SLOPE, 
#                 negative_slope=NEGATIVE_SLOPE,
#                 starting_hour=9,
#                 ending_hour=18,
#                 lookback_window=LABEL_LOOKBACK)


# os.makedirs(f"{working_path}/{instrument}", exist_ok=True)

# hour_df.to_csv(f"{working_path}/{instrument}/hour.csv", index=False)
# split_multiresolution_chunks(df_5min=df,
#                             df_hour=hour_df,
#                             dump_path=f"{working_path}/{instrument}",
#                             chunk_size=20000,
#                             hour_lookback=OTHER_TOKENS,
#                             lookback=NUM_TOKENS,
#                             cols=[
#                                 'time',
#                                 'position_in_hour',
#                                 'partial_hour_length',
#                                 'open_normalized',
#                                 'high_normalized',
#                                 'low_normalized',
#                                 'close_normalized',
#                                 'rsi',
#                                 'partial_open_normalized',
#                                 'partial_high_normalized',
#                                 'partial_low_normalized',
#                                 'partial_close_normalized',
#                                 'partial_rsi',
#                                 'include',
#                                 'target_high',
#                                 'target_low'
#                             ])

In [5]:
import os
from generators.regression_multi_instrument_data_generator import InstrumentConfig, MultiInstrumentDatasetConfig, create_multi_instrument_dataset
from constants.global_constants import FEATURES, NUM_TOKENS, OTHER_TOKENS, BATCH_SIZE, LOOKBACK_WINDOW


instruments = os.listdir(working_path)
instruments = [instrument]
feature_cols = FEATURES + ['rsi']

def get_datasets_and_steps(instruments=instruments, working_path=working_path, feature_cols=feature_cols):
    train_instrument_configs = []
    val_instrument_configs = []
    test_instrument_configs = []

    for instrument in instruments:
        train_instrument_configs.append(
            InstrumentConfig(
                name=instrument,
                hourly_data_path=f"{working_path}/{instrument}/hour.csv",
                chunked_data_dir=f"{working_path}/{instrument}/training"
            )
        )
        val_instrument_configs.append(
            InstrumentConfig(
                name=instrument,
                hourly_data_path=f"{working_path}/{instrument}/hour.csv",
                chunked_data_dir=f"{working_path}/{instrument}/validation"
            )
        )
        test_instrument_configs.append(
            InstrumentConfig(
                name=instrument,
                hourly_data_path=f"{working_path}/{instrument}/hour.csv",
                chunked_data_dir=f"{working_path}/{instrument}/testing"
            )
        )

    train_config = MultiInstrumentDatasetConfig(
        instruments=train_instrument_configs,
        main_lookback_tokens=NUM_TOKENS,
        hourly_lookback_tokens=OTHER_TOKENS,
        lookback_window=LOOKBACK_WINDOW,
        batch_size=BATCH_SIZE,
        shuffle_data=True,
        feature_columns=feature_cols,
        max_chunks_per_instrument=25
    )

    val_config = MultiInstrumentDatasetConfig(
        instruments=val_instrument_configs,
        main_lookback_tokens=NUM_TOKENS,
        hourly_lookback_tokens=OTHER_TOKENS,
        lookback_window=LOOKBACK_WINDOW,
        batch_size=BATCH_SIZE,
        shuffle_data=False,
        feature_columns=feature_cols,
        max_chunks_per_instrument=25
    )

    test_config = MultiInstrumentDatasetConfig(
        instruments=test_instrument_configs,
        main_lookback_tokens=NUM_TOKENS,
        hourly_lookback_tokens=OTHER_TOKENS,
        lookback_window=LOOKBACK_WINDOW,
        batch_size=BATCH_SIZE,
        shuffle_data=False,
        feature_columns=feature_cols,
        max_chunks_per_instrument=25
    )


    train_dataset, train_rows = create_multi_instrument_dataset(
        config=train_config,
        repeat_dataset=True
    )
    val_dataset, val_rows = create_multi_instrument_dataset(
        config=val_config,
        repeat_dataset=True
    )
    test_dataset, test_rows = create_multi_instrument_dataset(
        config=test_config,
        repeat_dataset=True
    )

    train_steps = train_rows//BATCH_SIZE
    val_steps = val_rows//BATCH_SIZE
    test_steps = test_rows//BATCH_SIZE

    return (
        (train_dataset, val_dataset, test_dataset),
        (train_steps, val_steps, test_steps)
    )

In [6]:
(train_dataset, val_dataset, test_dataset), (train_steps, val_steps, test_steps) = get_datasets_and_steps()

2025-07-24 20:13:31,706 - INFO - Discovered 86 chunk files for GBPUSD#
2025-07-24 20:13:31,707 - INFO - Loading hourly data for GBPUSD#...
2025-07-24 20:13:31,727 - INFO - Loaded hourly data for GBPUSD#: 143866 rows
2025-07-24 20:13:31,730 - INFO - Applied time threshold for GBPUSD#: 979318800
2025-07-24 20:13:31,731 - INFO - Building indices for GBPUSD#...
2025-07-24 20:13:34,481 - INFO - Built indices for GBPUSD#: 79738 valid samples
2025-07-24 20:13:34,482 - INFO - Building global indices across all instruments...
2025-07-24 20:13:34,488 - INFO - Global indices built: 79738 total samples across 1 instruments
2025-07-24 20:13:34,488 - INFO - === Multi-Instrument Regression Dataset Info ===
2025-07-24 20:13:34,489 - INFO - Total instruments: 1
2025-07-24 20:13:34,489 - INFO -   GBPUSD#: 79738 samples, 86 chunks
2025-07-24 20:13:34,490 - INFO - Global total: 79738 samples
2025-07-24 20:13:34,490 - INFO - Target columns: ['target_high', 'target_low']
2025-07-24 20:13:34,491 - INFO - Shu

In [7]:
def get_naive_baseline_metrics(val_dataset, val_steps):
    """
    Calculate naive baseline metrics for target_high predictions.
    Uses mean prediction as the naive baseline.
    
    Args:
        val_dataset: TensorFlow dataset from create_multi_instrument_dataset
        val_steps: Number of validation steps/batches to process
        
    Returns:
        dict: Contains baseline_value, mae, mse, rmse
    """
    # Collect all target_high values
    all_target_highs = []
    
    for i, batch in enumerate(val_dataset):
        if i >= val_steps:
            break
        (main_input, hourly_input, partial, position, hourly_position), targets = batch
        target_highs = targets['target_high'].numpy()
        all_target_highs.extend(target_highs)
    
    all_target_highs = np.array(all_target_highs)
    
    # Calculate baseline (mean of all targets)
    baseline_value = np.mean(all_target_highs)
    
    # Create predictions (always predict the mean)
    predictions = np.full_like(all_target_highs, baseline_value)
    
    # Calculate metrics
    mae = np.mean(np.abs(predictions - all_target_highs))
    mse = np.mean((predictions - all_target_highs) ** 2)
    rmse = np.sqrt(mse)
    
    return {
        'baseline_value': baseline_value,
        'mae': mae,
        'mse': mse,
        'rmse': rmse,
        'total_samples': len(all_target_highs)
    }

In [8]:
# get_naive_baseline_metrics(val_dataset, val_steps)

In [9]:
model = create_regression_model(feature_cols=feature_cols, d_model=R_D_MODEL, num_heads=R_NUM_HEADS, ff_dim=R_FF_DIM,
                                num_tokens=NUM_TOKENS, other_tokens=OTHER_TOKENS)
model = compile_model_lightweight(model=model)
print(model.metrics_names)

['loss', 'compile_metrics']


In [10]:
history = model.fit(
    train_dataset,
    epochs=50,
    steps_per_epoch=train_steps,
    validation_data=val_dataset,
    validation_steps=val_steps
)

Epoch 1/50


I0000 00:00:1753380833.678723   94154 service.cc:145] XLA service 0x7f577c003990 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1753380833.678858   94154 service.cc:153]   StreamExecutor device (0): NVIDIA GeForce RTX 3070 Ti Laptop GPU, Compute Capability 8.6
2025-07-24 20:13:54.012760: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
W0000 00:00:1753380834.032568   94154 random_ops.cc:59] Warning: Using tf.random.uniform with XLA compilation will ignore seeds; consider using tf.random.stateless_uniform instead if reproducible behavior is desired. functional_1/stochastic_gated_transformer_block_1/random_uniform/RandomUniform
2025-07-24 20:13:57.279660: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:465] Loaded cuDNN version 8907
I0000 00:00:1753380854.160574   94301 asm_compiler.cc:369] ptxas warning : Registers are

1245/1245 ━━━━━━━━━━━━━━━━━━━━ 0s 288ms/step - loss: 13.1799 - target_high_loss: 7.7833 - target_high_mae: 4.0801 - target_high_metric: 0.7181 - target_high_metric_1: 0.1683 - target_high_metric_2: 0.1920 - target_high_mse: 33.6047 - target_low_loss: 5.3966 - target_low_mae: 2.7677 - target_low_metric: 0.4563

I0000 00:00:1753381238.371486   95737 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_96', 16 bytes spill stores, 16 bytes spill loads

I0000 00:00:1753381238.675946   95739 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_96', 168 bytes spill stores, 168 bytes spill loads

I0000 00:00:1753381238.828007   95722 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_96', 320 bytes spill stores, 304 bytes spill loads

I0000 00:00:1753381239.127785   95730 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_96', 300 bytes spill stores, 284 bytes spill loads

I0000 00:00:1753381239.140754   95725 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_96', 20 bytes spill stores, 20 bytes spill loads

I0000 00:00:1753381239.779255   95

1245/1245 ━━━━━━━━━━━━━━━━━━━━ 462s 323ms/step - loss: 13.1784 - target_high_loss: 7.7821 - target_high_mae: 4.0797 - target_high_metric: 0.7181 - target_high_metric_1: 0.1683 - target_high_metric_2: 0.1921 - target_high_mse: 33.5970 - target_low_loss: 5.3963 - target_low_mae: 2.7675 - target_low_metric: 0.4563 - val_loss: 10.5193 - val_target_high_loss: 5.8154 - val_target_high_mae: 3.0393 - val_target_high_metric: 0.8243 - val_target_high_metric_1: 0.0473 - val_target_high_metric_2: 0.0096 - val_target_high_mse: 17.3537 - val_target_low_loss: 4.7039 - val_target_low_mae: 2.2802 - val_target_low_metric: 0.4547
Epoch 2/50
1245/1245 ━━━━━━━━━━━━━━━━━━━━ 0s 253ms/step - loss: 10.4378 - target_high_loss: 5.6141 - target_high_mae: 3.4431 - target_high_metric: 0.7390 - target_high_metric_1: 0.2962 - target_high_metric_2: 0.3475 - target_high_mse: 20.9543 - target_low_loss: 4.8238 - target_low_mae: 2.5129 - target_low_metric: 0.4526

I0000 00:00:1753381597.234311   96862 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_42', 4 bytes spill stores, 4 bytes spill loads

I0000 00:00:1753381597.998105   96862 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_42', 4 bytes spill stores, 4 bytes spill loads



1245/1245 ━━━━━━━━━━━━━━━━━━━━ 358s 288ms/step - loss: 10.4379 - target_high_loss: 5.6141 - target_high_mae: 3.4431 - target_high_metric: 0.7390 - target_high_metric_1: 0.2963 - target_high_metric_2: 0.3475 - target_high_mse: 20.9545 - target_low_loss: 4.8238 - target_low_mae: 2.5129 - target_low_metric: 0.4526 - val_loss: 10.6523 - val_target_high_loss: 6.0023 - val_target_high_mae: 2.8311 - val_target_high_metric: 0.8160 - val_target_high_metric_1: 0.2623 - val_target_high_metric_2: 0.1042 - val_target_high_mse: 16.4049 - val_target_low_loss: 4.7610 - val_target_low_mae: 2.1423 - val_target_low_metric: 0.4545
Epoch 3/50
1245/1245 ━━━━━━━━━━━━━━━━━━━━ 359s 288ms/step - loss: 10.3712 - target_high_loss: 5.5684 - target_high_mae: 3.4456 - target_high_metric: 0.7378 - target_high_metric_1: 0.3149 - target_high_metric_2: 0.3932 - target_high_mse: 21.1168 - target_low_loss: 4.8028 - target_low_mae: 2.5146 - target_low_metric: 0.4517 - val_loss: 10.0805 - val_target_high_loss: 5.5679 - val_